# Software Design

A structured reference covering architectural patterns, design principles, programming paradigms, and cross-cutting concerns — from awareness to expert application.

---

## 1. Architectural Patterns

Architectural patterns define the high-level structure of a system.

| Pattern | Description | Best For |
|---|---|---|
| **Layered** | Divides system into horizontal layers (Presentation → Business → Data) | Enterprise apps, clear separation of concerns |
| **Client-Server** | Clients request services from a centralized server | Web apps, APIs |
| **MVC** | Model (data), View (UI), Controller (logic bridge) | UI-heavy apps |
| **SOA** | Services communicate via a shared bus/protocol | Legacy enterprise integration |
| **Event-Driven** | Components communicate through events asynchronously | Real-time systems, decoupled microservices |
| **Microservices** | Small, independently deployable services per business domain | Scalable, cloud-native apps |
| **Micro-frontends** | Frontend decomposed into independent feature modules | Large teams, multi-framework UIs |

**When to choose:** Event-driven suits high-throughput async workloads (e.g., order processing); Microservices suit teams needing independent deployability; Layered suits smaller, cohesive apps.

---

## 2. Design Patterns (GoF & Beyond)

### Creational
Control object creation.

- **Singleton** — Ensures only one instance exists.
```python
  class Config:
      _instance = None

      def __new__(cls):
          if cls._instance is None:
              cls._instance = super().__new__(cls)
              cls._instance.settings = {"theme": "dark", "lang": "en"}
          return cls._instance

  # Only one config instance across the app
  a = Config()
  b = Config()
  print(a is b)          # → True
  print(a.settings)      # → {'theme': 'dark', 'lang': 'en'}
```

- **Factory Method** — Delegates instantiation to subclasses.
```python
  from abc import ABC, abstractmethod

  class Shape(ABC):
      @abstractmethod
      def draw(self) -> str: ...

  class Circle(Shape):
      def draw(self) -> str: return "Drawing a Circle"

  class Square(Shape):
      def draw(self) -> str: return "Drawing a Square"

  class ShapeFactory:
      @staticmethod
      def create(shape_type: str) -> Shape:
          shapes = {"circle": Circle, "square": Square}
          cls = shapes.get(shape_type.lower())
          if not cls:
              raise ValueError(f"Unknown shape: {shape_type}")
          return cls()

  shape = ShapeFactory.create("circle")
  print(shape.draw())    # → Drawing a Circle
```

### Structural
Compose objects and classes.

- **Decorator** — Adds behavior without altering the original class.
```python
  from abc import ABC, abstractmethod

  class TextReader(ABC):
      @abstractmethod
      def read(self) -> str: ...

  class FileReader(TextReader):
      def __init__(self, filename: str):
          self._filename = filename
      def read(self) -> str:
          return f"raw content of {self._filename}"

  class BufferedReader(TextReader):
      def __init__(self, reader: TextReader):
          self._reader = reader
      def read(self) -> str:
          return f"[buffered] {self._reader.read()}"

  class EncryptedReader(TextReader):
      def __init__(self, reader: TextReader):
          self._reader = reader
      def read(self) -> str:
          return f"[encrypted] {self._reader.read()}"

  # Stack decorators at runtime
  reader = EncryptedReader(BufferedReader(FileReader("file.txt")))
  print(reader.read())
  # → [encrypted] [buffered] raw content of file.txt
```

### Behavioral
Manage algorithms and responsibilities.

- **Observer** — Notifies dependents of state changes (e.g., event listeners).
- **Strategy** — Swaps algorithms at runtime.
```python
  from abc import ABC, abstractmethod

  class SortStrategy(ABC):
      @abstractmethod
      def sort(self, data: list) -> list: ...

  class QuickSort(SortStrategy):
      def sort(self, data: list) -> list:
          if len(data) <= 1:
              return data
          pivot = data[len(data) // 2]
          left = [x for x in data if x < pivot]
          mid  = [x for x in data if x == pivot]
          right = [x for x in data if x > pivot]
          return self.sort(left) + mid + self.sort(right)

  class BubbleSort(SortStrategy):
      def sort(self, data: list) -> list:
          data = data[:]
          for i in range(len(data)):
              for j in range(len(data) - i - 1):
                  if data[j] > data[j + 1]:
                      data[j], data[j + 1] = data[j + 1], data[j]
          return data

  class Sorter:
      def __init__(self, strategy: SortStrategy):
          self._strategy = strategy
      def set_strategy(self, strategy: SortStrategy):
          self._strategy = strategy
      def sort(self, data: list) -> list:
          return self._strategy.sort(data)

  # Usage
  sorter = Sorter(QuickSort())
  print(sorter.sort([3, 1, 4, 1, 5]))  # → [1, 1, 3, 4, 5]

  sorter.set_strategy(BubbleSort())
  print(sorter.sort([3, 1, 4, 1, 5]))  # → [1, 1, 3, 4, 5]
```

### Dependency Injection / IoC
Inverts control by injecting dependencies from outside.
```python
# Instead of: service = Service()
class Service:
    def fetch(self) -> str:
        return "real data"

class MockService:
    def fetch(self) -> str:
        return "mock data"

class Controller:
    def __init__(self, service: Service):  # dependency injected from outside
        self._service = service

# Production
controller = Controller(Service())
print(controller._service.fetch())       # → real data

# Testing
controller = Controller(MockService())
print(controller._service.fetch())       # → mock data
```
Promotes testability and loose coupling.

### Integration & Messaging Patterns
- **Message Queue** — Decouples producers/consumers (e.g., RabbitMQ, Kafka).
- **Saga Pattern** — Manages distributed transactions across microservices.
- **API Gateway** — Single entry point for routing, auth, rate-limiting.
- **Circuit Breaker** — Stops cascading failures in distributed systems.

---

## 3. Design Principles

### Basic Principles (KISS, DRY, YAGNI)
| Principle | Meaning | Example |
|---|---|---|
| **KISS** | Keep It Simple, Stupid — avoid unnecessary complexity | Prefer a clear `if/else` over a complex regex |
| **DRY** | Don't Repeat Yourself — single source of truth | Extract duplicated logic into a shared utility |
| **YAGNI** | You Aren't Gonna Need It — don't build for hypothetical future | Skip that "flexible plugin system" if no one asked |

### SOLID
| Letter | Principle | Short Rule |
|---|---|---|
| **S** | Single Responsibility | One class = one reason to change |
| **O** | Open/Closed | Open for extension, closed for modification |
| **L** | Liskov Substitution | Subtypes must be replaceable for their base type |
| **I** | Interface Segregation | Many specific interfaces > one general interface |
| **D** | Dependency Inversion | Depend on abstractions, not concretions |

### GRASP
Assigns responsibilities to classes/objects with patterns like:
- **Information Expert** — Give responsibility to the class that has the data.
- **Low Coupling / High Cohesion** — Classes should do one thing well and depend on few others.
- **Controller** — Route system events to appropriate handlers.

### Clean Code Principles
- Meaningful names: `calculateTax()` not `calc()`
- Small functions with a single purpose
- No magic numbers: use named constants
- Fail fast and be explicit with errors

---

## 4. Programming Paradigms

### Object-Oriented Programming (OOP)
Organizes code around **objects** with state and behavior.
- **Pros:** Intuitive modeling, encapsulation, reuse via inheritance/composition
- **Cons:** Can lead to complex hierarchies, harder to parallelize
- **Use when:** Modeling real-world entities (e.g., `User`, `Order`)
```python
from dataclasses import dataclass, field
from datetime import datetime

@dataclass
class User:
    name: str
    email: str
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())

    def greet(self) -> str:
        return f"Hello, {self.name}!"

@dataclass
class Order:
    user: User
    items: list[str]
    total: float

    def summary(self) -> str:
        return (f"Order for {self.user.name}: "
                f"{', '.join(self.items)} — ${self.total:.2f}")

# Usage
user  = User(name="Ana", email="ana@example.com")
order = Order(user=user, items=["Book", "Pen"], total=24.99)

print(user.greet())       # → Hello, Ana!
print(order.summary())    # → Order for Ana: Book, Pen — $24.99
```

### Functional Programming (FP)
Treats computation as **pure functions**, avoids shared state.
```python
from functools import reduce

numbers = [1, 2, 3, 4, 5]

# map — transform each element (no mutation)
doubled = list(map(lambda x: x * 2, numbers))
print(doubled)                        # → [2, 4, 6, 8, 10]

# filter — keep elements matching a condition
evens = list(filter(lambda x: x % 2 == 0, numbers))
print(evens)                          # → [2, 4]

# reduce — fold list into a single value
total = reduce(lambda acc, x: acc + x, numbers)
print(total)                          # → 15

# chaining with pure functions
result = reduce(
    lambda acc, x: acc + x,
    filter(lambda x: x % 2 == 0,
    map(lambda x: x * 2, numbers))
)
print(result)                         # → 12  (4 + 8)
```
- **Pros:** Predictable, testable, parallelizable
- **Cons:** Steeper learning curve, less intuitive for stateful UIs
- **Use when:** Data transformation pipelines, concurrent systems

### Reactive Programming (RP)
Models data as **streams** that propagate changes.
```python
from rx import create, operators as ops
import time, threading

def keyup_stream(observer, scheduler):
    keystrokes = ["h", "he", "hel", "hell", "hello"]
    for key in keystrokes:
        time.sleep(0.1)
        observer.on_next(key)
    observer.on_completed()

def search(query: str):
    print(f"Searching for: {query}")

# Create stream → debounce → subscribe
source = create(keyup_stream)
source.pipe(
    ops.debounce(0.3),        # emit only after 0.3s pause
).subscribe(on_next=search)

time.sleep(2)                 # → Searching for: hello
```
- **Pros:** Elegant async handling, responsive UIs
- **Cons:** Debugging complexity, steep learning curve
- **Use when:** Real-time UIs, event-heavy systems

> In practice, paradigms are **mixed**: OOP for structure, FP for transformations, RP for async flows.

---

## 5. Cross-Cutting Concerns

Concerns that span multiple layers and modules — best handled centrally.

### Logging
- Use structured logging (JSON) with levels (`DEBUG`, `INFO`, `ERROR`).
- Avoid logging sensitive data (PII, tokens).
- Libraries: `Log4j`, `Serilog`, `Winston`.

### Error Handling
- Fail fast, fail clearly.
- Use typed exceptions/errors; avoid swallowing exceptions silently.
- Centralize handling: middleware (web), global handlers (apps).
```python
try:
    process_order(order)
except InsufficientStockError as e:
    log.warning("Stock issue", extra={"order_id": order.id})
    raise
```

### Secure Coding
- Validate and sanitize all inputs.
- Use parameterized queries (prevent SQL injection).
- Apply least privilege; never hardcode secrets.
- Leverage frameworks: OWASP guidelines, `Helmet.js`, Spring Security.

### Other Common Concerns
| Concern | Approach |
|---|---|
| **Caching** | Redis, in-memory cache with TTL |
| **Authentication/AuthZ** | OAuth2, JWT, RBAC |
| **Observability** | Metrics (Prometheus), tracing (OpenTelemetry) |
| **Configuration** | Externalized config (env vars, Vault) |

---

## 6. Technical Documentation

Good documentation is a **first-class artifact** of software design.

| Type | Purpose | Examples |
|---|---|---|
| **How-to guides** | Step-by-step task completion | "How to add a new endpoint" |
| **Coding standards** | Team-wide consistency rules | Naming conventions, formatter config |
| **Architecture diagrams** | Visualize system structure | C4 model, UML sequence/component |
| **ADRs** | Record architectural decisions with context | "Why we chose Kafka over RabbitMQ" |

**Tips:**
- Keep docs close to code (e.g., `/docs` folder, README files).
- Use diagrams-as-code tools: `Mermaid`, `PlantUML`, `Structurizr`.
- Update docs as part of the Definition of Done.

---

## 7. Applying Design on Solution Layers

Design patterns appear at every layer:

| Layer | Common Patterns | Purpose / Notes |
|---|---|---|
| **Presentation** | MVC, MVP, MVVM, Observer | Separate UI state from rendering logic; Observer drives reactive updates when model changes |
| **Application/Use Case** | Command, Mediator, Facade | Encapsulate user actions (Command), decouple orchestration logic (Mediator), simplify complex subsystem APIs (Facade) |
| **Domain** | Entity, Aggregate, Strategy, Specification | Model business concepts (Entity/Aggregate), swap business rules at runtime (Strategy), compose queryable business rules (Specification) |
| **Infrastructure** | Repository, Adapter, Factory, Singleton (DB pool) | Abstract data access (Repository), wrap third-party APIs (Adapter), centralise object creation (Factory), share expensive resources (Singleton) |
| **Cross-layer** | Decorator (logging/caching), DI/IoC, Circuit Breaker | Add concerns without touching core logic (Decorator), invert dependencies for testability (DI/IoC), prevent cascading failures to external services (Circuit Breaker) ||

**Example:** In a microservice, the HTTP Controller (MVC) delegates to a Use Case (Command pattern), which calls a Repository (Repository pattern) that wraps an ORM (Adapter pattern). DI wires it all together.

---

## Quick Reference: Decision Heuristics

| Symptom / Question | First Move | Patterns & Tactics to Reach For | Watch Out For |
|---|---|---|---|
| **Tight coupling?** | Introduce an abstraction boundary | DI/IoC, Repository, Facade, Adapter | Over-engineering simple glue code that doesn't actually change |
| **Duplicated logic?** | Identify the single source of truth | DRY + extract to shared module/service, Template Method, Strategy | Premature abstraction — wait for the third repetition (Rule of Three) |
| **Unpredictable growth?** | Build the simplest thing that works | YAGNI first; defer to Factory / Builder / Plugin when variation actually arrives | Gold-plating — hooks for requirements that never materialise |
| **Complex async flows?** | Make data flow explicit and traceable | Event-driven + Pub/Sub, Saga, Message Queue, Outbox Pattern | Hidden ordering dependencies; ensure idempotency and dead-letter handling |
| **Multiple teams on one system?** | Define hard ownership boundaries | Microservices, Micro-frontends, Bounded Contexts (DDD), API Gateway | Distributed monolith — splitting deployment without splitting the data model |
| **Need resilience?** | Isolate failure domains | Circuit Breaker, Retry + Backoff, Bulkhead, Health Check | Retry storms; pair every retry with jitter and a timeout budget |
| **Need scalability?** | Separate read and write paths | CQRS, Read Replicas, Caching (aside/write-through), Sharding | Cache invalidation complexity; eventual consistency surprises in the UI |
| **Slow or unreadable conditionals?** | Replace branching with polymorphism | Strategy, State, Chain of Responsibility, Rules Engine | Unnecessary indirection when a simple `if` is already clear |
| **Cross-cutting concerns leaking everywhere?** | Push concerns to the edges | Decorator, Middleware pipeline, AOP, Interceptor | Obscured control flow — make sure stack traces still point to the real culprit |